# Advanced Pitch Statistics from Statcast

This notebook explores the advanced pitch-level data used to build pitcher and batter profiles.

**Pitcher Arsenal Stats:**
- Velocity, spin, movement by pitch type
- Pitch usage percentages
- Whiff%, CSW%, zone%, chase induced

**Batter Profile Stats:**
- Performance vs pitch types (fastball, breaking, offspeed)
- Performance vs velocity buckets (90-93, 94-96, 97+)
- Performance vs movement profiles (high rise, heavy sweep, heavy drop)
- Contact quality (exit velo, xwOBA)

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from mlb_data import (
    get_statcast_pitches,
    get_pitcher_arsenal,
    get_pitcher_pitch_type_stats,
    get_batter_pitch_stats,
    get_batter_pitch_type_stats,
    get_plate_appearances,
)

# Import config
from src.config import (
    DATA_START,
    DATA_END,
    DATA_DIR,
)

pd.set_option('display.max_columns', 50)

print(f"Date range: {DATA_START} to {DATA_END}")

Date range: 2021-04-01 to 2026-04-09


In [2]:
# Memory management utilities
import gc
import psutil

def clear_mem():
    """Run garbage collection and print memory usage."""
    gc.collect()
    mem_gb = psutil.Process().memory_info().rss / 1024**3
    print(f"Memory: {mem_gb:.2f} GB")
    return mem_gb

clear_mem()

Memory: 0.18 GB


0.18269729614257812

## 1. Raw Statcast Data

First, let's fetch the raw pitch-level data and explore what's available.

In [3]:
from pathlib import Path

# Load pitches from notebook 01 output
pitches_path = Path(f"../{DATA_DIR}/raw/pitches.parquet")

if pitches_path.exists():
    pitches = pd.read_parquet(pitches_path)
    print(f"Loaded pitches from {pitches_path}")
else:
    print(f"Pitches file not found at {pitches_path}")
    print("Run notebook 01_compile_data first to fetch the data.")
    raise FileNotFoundError(f"Run notebook 01 first: {pitches_path}")

print(f"Shape: {pitches.shape}")
print(f"Date range: {pitches['game_date'].min()} to {pitches['game_date'].max()}")
pitches.head()

clear_mem()

Loaded pitches from ../data/raw/pitches.parquet
Shape: (3896176, 118)
Date range: 2021-04-01 00:00:00 to 2026-04-08 00:00:00
Memory: 1.23 GB


1.2280654907226562

In [4]:
# Key pitch characteristic columns
pitch_cols = [
    'pitch_type', 'release_speed', 'release_spin_rate',
    'pfx_x', 'pfx_z', 'release_extension',
    'plate_x', 'plate_z', 'zone',
    'description', 'events', 'type',
]

pitches[pitch_cols].head(20)


clear_mem()

Memory: 0.79 GB


0.786224365234375

In [5]:
# Pitch type distribution
print("Pitch type distribution:")
pitch_counts = pitches['pitch_type'].value_counts()
pitch_pcts = pitches['pitch_type'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': pitch_counts,
    'pct': pitch_pcts.round(1)
}).head(15)

Pitch type distribution:


,count,pct
pitch_type,,
FF,1254195,32.9
SL,599345,15.7
SI,592157,15.5
CH,409228,10.7
FC,293496,7.7
CU,260746,6.8
ST,206746,5.4
FS,90605,2.4
KC,76740,2.0


In [6]:
# Pitch outcome descriptions
print("Pitch descriptions (outcomes):")
pitches['description'].value_counts()

Pitch descriptions (outcomes):


description
ball                       1293728
foul                        687897
hit_into_play               683364
called_strike               634082
swinging_strike             418606
blocked_ball                 84767
foul_tip                     38373
swinging_strike_blocked      23184
automatic_ball               12009
hit_by_pitch                 11389
foul_bunt                     6531
missed_bunt                   1242
automatic_strike               628
pitchout                       237
bunt_foul_tip                  136
foul_pitchout                    2
intent_ball                      1
Name: count, dtype: int64

## 2. Pitcher Arsenal Profiles

Aggregate pitch-level data into pitcher profiles with:
- Fastball characteristics (velocity, spin, movement)
- Breaking ball characteristics
- Offspeed characteristics
- Overall effectiveness metrics (whiff%, CSW%, etc.)

In [7]:
# Get pitcher arsenal stats (uses cached pitches)
pitcher_arsenal = get_pitcher_arsenal(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Pitchers with arsenal data: {len(pitcher_arsenal)}")
pitcher_arsenal.head(10)

clear_mem()

Computing pitcher arsenal stats...
  Computed arsenal for 1622 pitchers
Pitchers with arsenal data: 1622
Memory: 0.33 GB


0.3293495178222656

In [8]:
# All available arsenal columns
print("Pitcher arsenal columns:")
print(pitcher_arsenal.columns.tolist())

Pitcher arsenal columns:
['pitcher_id', 'pitcher_name', 'p_throws', 'total_pitches', 'primary_pitch', 'ff_usage_pct', 'si_usage_pct', 'si_velo_avg', 'si_spin_avg', 'si_vmov_avg', 'si_hmov_avg', 'si_whiff_pct', 'fc_usage_pct', 'sl_usage_pct', 'sl_velo_avg', 'sl_spin_avg', 'sl_vmov_avg', 'sl_hmov_avg', 'sl_whiff_pct', 'cu_usage_pct', 'ch_usage_pct', 'sv_usage_pct', 'fs_usage_pct', 'kc_usage_pct', 'st_usage_pct', 'kn_usage_pct', 'whiff_pct', 'swstr_pct', 'csw_pct', 'zone_pct', 'chase_pct_induced', 'contact_pct', 'f_strike_pct', 'ff_velo_avg', 'ff_spin_avg', 'ff_vmov_avg', 'ff_hmov_avg', 'ff_whiff_pct', 'fc_velo_avg', 'fc_spin_avg', 'fc_vmov_avg', 'fc_hmov_avg', 'fc_whiff_pct', 'cu_velo_avg', 'cu_spin_avg', 'cu_vmov_avg', 'cu_hmov_avg', 'cu_whiff_pct', 'ch_velo_avg', 'ch_spin_avg', 'ch_vmov_avg', 'ch_hmov_avg', 'ch_whiff_pct', 'st_velo_avg', 'st_spin_avg', 'st_vmov_avg', 'st_hmov_avg', 'st_whiff_pct', 'fs_velo_avg', 'fs_spin_avg', 'fs_vmov_avg', 'fs_hmov_avg', 'fs_whiff_pct', 'kc_velo_avg'

In [9]:
# Summary statistics for key arsenal metrics
arsenal_stats = [
    'fb_velo_avg', 'fb_velo_max', 'fb_spin_avg', 'fb_vmov_avg',
    'brk_velo_avg', 'brk_hmov_avg', 'brk_vmov_avg',
    'off_velo_avg', 'off_velo_diff',
    'fb_usage_pct', 'brk_usage_pct', 'off_usage_pct',
    'whiff_pct', 'csw_pct', 'zone_pct', 'chase_pct_induced',
]

available_stats = [c for c in arsenal_stats if c in pitcher_arsenal.columns]
pitcher_arsenal[available_stats].describe().round(2)

,whiff_pct,csw_pct,zone_pct,chase_pct_induced
count,1622.00,1622.00,1622.00,1622.00
mean,0.23,0.27,0.48,0.27
std,0.06,0.04,0.04,0.05
min,0.00,0.06,0.17,0.03
25%,0.19,0.25,0.46,0.24
50%,0.23,0.27,0.49,0.27
75%,0.26,0.29,0.51,0.30
max,0.53,0.38,0.68,0.45


## 3. Pitcher Stats by Pitch Type

Detailed breakdown for each pitch type a pitcher throws.

In [10]:
# Get per-pitch-type stats for pitchers
pitcher_pitch_types = get_pitcher_pitch_type_stats(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Pitcher-pitch type combinations: {len(pitcher_pitch_types)}")
pitcher_pitch_types.head(15)

Computing pitcher pitch-type stats...
  Computed stats for 5555 pitcher-pitch combinations
Pitcher-pitch type combinations: 5555


,pitcher_id,pitcher_name,p_throws,pitch_type,pitch_count,usage_pct,velo_avg,velo_max,spin_avg,vmov_avg,hmov_avg,extension,whiff_pct,csw_pct,swing_pct,avg_exit_velo,avg_launch_angle,xwoba_against,xba_against
0,424144,"Pérez, Oliver",L,SI,82,0.569444,88.632927,91.9,2107.890244,0.643780,1.493171,6.219512,0.113636,0.231707,0.536585,88.014815,9.407407,0.379178,0.358842
1,425794,"Wainwright, Adam",R,CH,495,0.060550,82.281414,85.7,1722.395918,0.668141,-1.169253,6.468024,0.160804,0.202020,0.402020,86.911765,4.901961,0.308929,0.279908
2,425794,"Wainwright, Adam",R,CU,2596,0.317554,72.850462,77.1,2774.716127,-1.211102,1.363532,6.322616,0.219780,0.318567,0.455701,85.488555,10.195122,0.363505,0.330751
3,425794,"Wainwright, Adam",R,FC,1889,0.231070,84.212176,89.8,2390.914654,0.546236,0.514002,6.469796,0.173333,0.246162,0.516146,86.548762,11.445545,0.369198,0.327909
4,425794,"Wainwright, Adam",R,FF,780,0.095413,87.859103,92.3,2216.473272,1.233038,-0.131449,6.537273,0.120879,0.253846,0.350000,90.236434,11.658915,0.414309,0.366921
5,425794,"Wainwright, Adam",R,SI,2358,0.288440,88.346735,92.8,2202.328879,0.995619,-1.079330,6.538518,0.075196,0.282867,0.377863,91.784807,11.705215,0.410986,0.355862
6,425844,"Greinke, Zack",R,CH,1364,0.187234,86.464516,90.2,1657.978629,0.345579,-1.074655,5.946188,0.214391,0.147361,0.499267,87.129487,-5.599359,0.299723,0.291535
7,425844,"Greinke, Zack",R,CU,1218,0.167193,71.815025,76.8,2401.838127,-0.903128,0.987693,5.844207,0.187500,0.268473,0.499179,85.312375,14.866221,0.350622,0.299783
8,425844,"Greinke, Zack",R,FC,505,0.069321,85.619406,90.7,2391.270378,0.459228,0.555703,5.924405,0.172549,0.209901,0.504950,87.741071,11.026786,0.401614,0.346655
9,425844,"Greinke, Zack",R,FF,2578,0.353878,89.166757,93.3,2256.949903,1.340559,-0.161462,5.978123,0.117702,0.318076,0.411947,91.989032,19.707527,0.435116,0.366007


In [11]:
# Best sliders by whiff rate
sliders = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'SL']
print("Top 10 sliders by whiff rate:")
sliders.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'hmov_avg', 'vmov_avg', 'whiff_pct'
]]

Top 10 sliders by whiff rate:


,pitcher_name,pitch_count,usage_pct,velo_avg,hmov_avg,vmov_avg,whiff_pct
4445,"Garcia, Luis",102,0.017152,85.039216,0.458725,0.036569,0.633333
463,"Ramírez, Erasmo",55,0.017805,81.480000,0.309455,0.391091,0.612903
3662,"Miller, Erik",146,0.075687,86.253425,-0.457397,-0.087329,0.606061
429,"Hendriks, Liam",655,0.261581,88.499695,0.196718,0.295802,0.591900
4425,"Crochet, Garrett",50,0.006782,86.346000,-0.612400,0.352800,0.588235
3383,"Felipe, Angel",96,0.354244,83.371875,0.258646,-0.388958,0.575758
4516,"Ferrer, Jose A.",161,0.067167,88.586335,-0.106398,0.159627,0.564516
1927,"Hader, Josh",1503,0.290659,84.393147,-0.240299,0.109894,0.557769
714,"Brothers, Rex",344,0.344689,86.179942,-0.085174,0.114302,0.533333
1682,"Reyes, Alex",353,0.281724,86.404816,0.688612,-0.023541,0.532051


In [12]:
# Best changeups by whiff rate
changeups = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'CH']
print("Top 10 changeups by whiff rate:")
changeups.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'vmov_avg', 'whiff_pct'
]]


# Clean up exploration variables
del sliders, changeups
clear_mem()

Top 10 changeups by whiff rate:
Memory: 0.36 GB


0.36081695556640625

## 4. Batter Pitch Performance Profiles

How batters perform against different pitch characteristics:
- By pitch type group (fastball, breaking, offspeed)
- By velocity bucket (90-93, 94-96, 97+)
- By movement type (high rise, heavy sweep, heavy drop)
- By pitcher handedness (vs LHP, vs RHP)

In [13]:
# Get batter pitch stats (uses cached pitches)
batter_profiles = get_batter_pitch_stats(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Batters with pitch profiles: {len(batter_profiles)}")
batter_profiles.head(10)

clear_mem()

Computing batter pitch stats...
  Computed pitch stats for 1293 batters
Batters with pitch profiles: 1293
Memory: 0.26 GB


0.2633628845214844

In [14]:
# All available batter profile columns
print("Batter profile columns:")
print(batter_profiles.columns.tolist())

Batter profile columns:
['batter_id', 'batter_name', 'stand', 'total_pitches', 'swing_pct', 'whiff_pct', 'contact_pct', 'csw_pct', 'zone_swing_pct', 'chase_pct', 'avg_exit_velo', 'max_exit_velo', 'avg_launch_angle', 'xwoba', 'xba', 'barrel_pct', 'vs_ff_whiff_pct', 'vs_ff_contact_pct', 'vs_ff_xwoba', 'vs_si_whiff_pct', 'vs_si_contact_pct', 'vs_si_xwoba', 'vs_fc_whiff_pct', 'vs_fc_contact_pct', 'vs_fc_xwoba', 'vs_sl_whiff_pct', 'vs_sl_contact_pct', 'vs_sl_xwoba', 'vs_cu_whiff_pct', 'vs_cu_contact_pct', 'vs_cu_xwoba', 'vs_ch_whiff_pct', 'vs_ch_contact_pct', 'vs_ch_xwoba', 'vs_kc_whiff_pct', 'vs_kc_contact_pct', 'vs_kc_xwoba', 'vs_st_whiff_pct', 'vs_st_contact_pct', 'vs_st_xwoba', 'velo_90_93_pitches', 'velo_90_93_whiff_pct', 'velo_90_93_swing_pct', 'velo_94_96_pitches', 'velo_94_96_whiff_pct', 'velo_94_96_swing_pct', 'velo_97_plus_pitches', 'velo_97_plus_whiff_pct', 'velo_97_plus_swing_pct', 'vs_LHP_pitches', 'vs_LHP_whiff_pct', 'vs_LHP_xwoba', 'vs_RHP_pitches', 'vs_RHP_whiff_pct', 'vs_RH

In [15]:
# Summary of key batter metrics
batter_stats = [
    'whiff_pct', 'contact_pct', 'chase_pct', 'zone_swing_pct',
    'fastball_whiff_pct', 'breaking_whiff_pct', 'offspeed_whiff_pct',
    'velo_97_plus_whiff_pct', 'high_rise_whiff_pct', 'heavy_sweep_whiff_pct',
    'avg_exit_velo', 'xwoba',
]

available_stats = [c for c in batter_stats if c in batter_profiles.columns]
batter_profiles[available_stats].describe().round(3)

,whiff_pct,contact_pct,chase_pct,zone_swing_pct,velo_97_plus_whiff_pct,avg_exit_velo,xwoba
count,1293.000,1293.000,1293.000,1293.000,845.000,1083.000,1083.000
mean,0.260,0.740,0.292,0.670,0.223,86.728,0.349
std,0.076,0.076,0.070,0.070,0.086,4.581,0.059
min,0.063,0.392,0.070,0.351,0.000,55.809,0.128
25%,0.211,0.697,0.245,0.632,0.164,85.337,0.312
50%,0.253,0.747,0.289,0.672,0.215,87.550,0.347
75%,0.303,0.789,0.332,0.715,0.273,89.330,0.387
max,0.608,0.937,0.681,0.900,0.562,95.926,0.593


In [16]:
# Best contact rates (lowest whiff)
print("Top 10 contact rates (lowest whiff%):")
batter_profiles.nsmallest(10, 'whiff_pct')[[
    'batter_id', 'stand', 'total_pitches',
    'whiff_pct', 'contact_pct', 'chase_pct', 'avg_exit_velo'
]]

Top 10 contact rates (lowest whiff%):


,batter_id,stand,total_pitches,whiff_pct,contact_pct,chase_pct,avg_exit_velo
550,650333,L,12104,0.063331,0.936669,0.296497,86.912380
214,591971,R,411,0.065359,0.934641,0.179104,84.010390
306,605200,R,153,0.066667,0.933333,0.115385,65.558333
1037,680757,L,11307,0.080824,0.919176,0.207036,84.865361
656,663611,R,3093,0.085106,0.914894,0.294194,84.802029
11,445055,L,54,0.085714,0.914286,0.518519,NaN
1279,805779,R,2298,0.092943,0.907057,0.340669,84.717007
1265,800325,L,87,0.093023,0.906977,0.238095,NaN
1271,802415,L,1745,0.095238,0.904762,0.298429,81.924368
741,665877,R,983,0.095718,0.904282,0.230769,84.467677


In [17]:
# Batters who struggle vs 97+ velocity
if 'velo_97_plus_whiff_pct' in batter_profiles.columns:
    has_velo_data = batter_profiles['velo_97_plus_whiff_pct'].notna()
    print("Batters who struggle most vs 97+ mph:")
    batter_profiles[has_velo_data].nlargest(10, 'velo_97_plus_whiff_pct')[[
        'batter_id', 'stand', 'velo_97_plus_pitches',
        'velo_97_plus_whiff_pct', 'whiff_pct'
    ]]

Batters who struggle most vs 97+ mph:


In [18]:
# Batters who struggle vs breaking balls
if 'breaking_whiff_pct' in batter_profiles.columns:
    has_brk_data = batter_profiles['breaking_whiff_pct'].notna()
    print("Batters who struggle most vs breaking balls:")
    batter_profiles[has_brk_data].nlargest(10, 'breaking_whiff_pct')[[
        'batter_id', 'stand', 'breaking_pitches',
        'breaking_whiff_pct', 'whiff_pct'
    ]]

## 5. Batter Stats by Pitch Type

Detailed breakdown for each pitch type a batter faces.

In [19]:
# Get per-pitch-type stats for batters
batter_pitch_types = get_batter_pitch_type_stats(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Batter-pitch type combinations: {len(batter_pitch_types)}")
batter_pitch_types.head(15)

Computing batter pitch-type stats...
  Computed stats for 7149 batter-pitch combinations
Batter-pitch type combinations: 7149


,batter_id,batter_name,stand,pitch_type,pitch_count,pct_of_pitches_seen,swing_pct,whiff_pct,contact_pct,avg_exit_velo,avg_launch_angle,xwoba,xba
0,405395,,R,CH,232,0.088550,0.461207,0.289720,0.710280,90.377778,4.361111,0.438562,0.370583
1,405395,,R,CU,231,0.088168,0.341991,0.354430,0.645570,85.900000,15.838710,0.322692,0.305167
2,405395,,R,FC,169,0.064504,0.526627,0.123596,0.876404,93.368421,12.210526,0.350156,0.296649
3,405395,,R,FF,777,0.296565,0.555985,0.115741,0.884259,92.598930,22.764706,0.415883,0.331442
4,405395,,R,KC,53,0.020229,0.377358,0.550000,0.450000,94.866667,4.666667,0.48062,0.427333
5,405395,,R,SI,456,0.174046,0.440789,0.129353,0.870647,92.335577,5.528846,0.35543,0.308843
6,405395,,R,SL,550,0.209924,0.463636,0.298039,0.701961,87.274444,6.544444,0.361531,0.277315
7,405395,,R,ST,108,0.041221,0.462963,0.260000,0.740000,82.161538,37.307692,0.183397,0.113583
8,408234,,R,CH,409,0.078428,0.486553,0.286432,0.713568,90.449367,4.468354,0.39906,0.355158
9,408234,,R,CU,252,0.048322,0.376984,0.305263,0.694737,87.730952,5.238095,0.331315,0.313341


## 6. Plate Appearance Outcomes

Extract completed plate appearances with outcomes (K, BB, 1B, 2B, 3B, HR, OUT).

In [21]:
# Get plate appearances with outcomes
plate_apps = get_plate_appearances(
    DATA_START, DATA_END,
    pitches_df=pitches
)

print(f"Total plate appearances: {len(plate_apps):,}")
plate_apps.head(15)

clear_mem()


# Free memory - pitches no longer needed after this point
del pitches
clear_mem()

Extracting 1,012,439 plate appearances...
Total plate appearances: 1,012,439
Memory: 0.77 GB
Memory: 0.45 GB


0.4510154724121094

In [22]:
# Outcome distribution
print("PA outcome distribution:")
outcome_counts = plate_apps['outcome'].value_counts()
outcome_pcts = plate_apps['outcome'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': outcome_counts,
    'pct': outcome_pcts.round(1)
})

PA outcome distribution:


,count,pct
outcome,,
OUT,461912,45.6
K,229972,22.7
1B,142650,14.1
BB,82811,8.2
2B,43943,4.3
HR,31083,3.1
HBP,11387,1.1
OTHER,4944,0.5
3B,3737,0.4


In [23]:
# Unique matchup counts
print(f"Unique pitchers: {plate_apps['pitcher_id'].nunique():,}")
print(f"Unique batters: {plate_apps['batter_id'].nunique():,}")

# Pitcher-batter combinations
matchups = plate_apps.groupby(['pitcher_id', 'batter_id']).size()
print(f"Unique pitcher-batter matchups: {len(matchups):,}")
print(f"\nPAs per matchup distribution:")
matchups.describe()

Unique pitchers: 2,803
Unique batters: 3,490
Unique pitcher-batter matchups: 366,922

PAs per matchup distribution:


count    366922.000000
mean          2.759276
std           2.927287
min           1.000000
25%           1.000000
50%           2.000000
75%           3.000000
max          53.000000
dtype: float64

## 7. Save Data

Save the computed profiles for use in model training.

In [24]:
from pathlib import Path

# Create output directory
output_dir = Path(f'../{DATA_DIR}/profiles')
output_dir.mkdir(parents=True, exist_ok=True)

# Save profiles
pitcher_arsenal.to_csv(output_dir / 'pitcher_arsenal.csv', index=False)
pitcher_pitch_types.to_csv(output_dir / 'pitcher_pitch_types.csv', index=False)
batter_profiles.to_csv(output_dir / 'batter_profiles.csv', index=False)
batter_pitch_types.to_csv(output_dir / 'batter_pitch_types.csv', index=False)
plate_apps.to_parquet(output_dir / 'plate_appearances.parquet', index=False)

print(f"Saved data to {output_dir.resolve()}:")
print(f"  - pitcher_arsenal.csv: {len(pitcher_arsenal):,} rows")
print(f"  - pitcher_pitch_types.csv: {len(pitcher_pitch_types):,} rows")
print(f"  - batter_profiles.csv: {len(batter_profiles):,} rows")
print(f"  - batter_pitch_types.csv: {len(batter_pitch_types):,} rows")
print(f"  - plate_appearances.parquet: {len(plate_apps):,} rows")

clear_mem()


# Clean up all dataframes
del pitcher_arsenal, batter_profiles, plate_apps
try:
    del pitcher_pitch_types, batter_pitch_types
except NameError:
    pass  # May not exist if those cells were skipped
clear_mem()

Saved data to /Users/matthewgillies/PitcherGamePreds/data/profiles:
  - pitcher_arsenal.csv: 1,622 rows
  - pitcher_pitch_types.csv: 5,555 rows
  - batter_profiles.csv: 1,293 rows
  - batter_pitch_types.csv: 7,149 rows
  - plate_appearances.parquet: 1,012,439 rows
Memory: 0.66 GB
Memory: 0.64 GB


0.6381340026855469